In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *


df = pd.read_csv("./data/pkaBasicData.csv")
df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

train_df = train_df.rename(columns={"pKa_Basic": "pKb"})
val_df = val_df.rename(columns={"pKa_Basic": "pKb"})
test_df = test_df.rename(columns={"pKa_Basic": "pKb"})

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="pKb",
    max_epochs=200
)

[07:50:40] WARNING: not removing hydrogen atom without neighbors
Seed set to 42


Training set size: 5201
Test set size: 650
Validation set size: 651


[07:50:41] WARNING: not removing hydrogen atom without neighbors
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



Training: BondMP3 + Mean


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'pKb', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.397260457277298, 'Test_RMSE': 0.7189713716506958, 'Test_R2': 0.9419757723808289}

Final comparison:
   Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
21    pKb        BondMP6      MultiHead4  0.353995   0.639823  0.954048
20    pKb        BondMP6      Attentive1  0.359853   0.646723  0.953051
15    pKb        BondMP5      MultiHead4  0.367249   0.643933  0.953456
22    pKb        BondMP6      MultiHead8  0.367956   0.651913  0.952295
19    pKb        BondMP6  GatedAttentive  0.371214   0.660681  0.951003
16    pKb        BondMP5      MultiHead8  0.372365   0.643032  0.953586
23    pKb        BondMP6     MultiHead12  0.374601   0.676699  0.948598
13    pKb        BondMP5  GatedAttentive  0.375933   0.677995  0.948401
43    pKb        AtomMP6  GatedAttentive  0.376120   0.736621  0.939092
11    pKb        BondMP4     MultiHead12  0.377552   0.677817  0.948428
8     pKb  